In [2]:
!pip install -q google-generativeai

import google.generativeai as genai
from getpass import getpass

# getpass keeps the key out of notebook output — important if you share the .ipynb
genai.configure(api_key=getpass("Paste your Gemini API key: "))

# Verify the key works before doing anything else
print(genai.GenerativeModel("gemma-4-26b-a4b-it").generate_content("Say OK").text.strip())

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Paste your Gemini API key: ··········
The user wants me to say the word "OK".
The request is simple and direct.
"OK" (or "Okay").
OK


In [3]:
import pandas as pd, os
from google.colab import files

uploaded = files.upload()          # abstract-only sheet
df = pd.read_excel(list(uploaded.keys())[0])

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

# ---- Alignment check ----
# paper_id is the row index, so row ORDER and COUNT must match the labelled
# run exactly, or the Cell E merge compares different papers to each other.
LABELLED_RESULTS = "results_gemini_labelled.csv"   # upload this too, or mount Drive
if os.path.exists(LABELLED_RESULTS):
    n_lab = pd.read_csv(LABELLED_RESULTS).paper_id.nunique()
    print(f"\nLabelled run covered {n_lab} papers | this sheet has {len(df)} rows")
    print("✓ Counts match" if n_lab == len(df)
          else "⚠️  MISMATCH — paper_id will not align. Fix before running.")
else:
    print("\n(Labelled results not uploaded — Cell E will be skipped)")

df.head(3)

Saving Unlabeled Dataset.xlsx to Unlabeled Dataset.xlsx
Rows: 300
Columns: ['Abstract']

(Labelled results not uploaded — Cell E will be skipped)


,Abstract
0,We study empirical scaling laws for language m...
1,The reliability of artificial intelligence hin...
2,"We trained a large, deep convolutional neural ..."


In [4]:
import google.generativeai as genai
import re, time, os, hashlib
import pandas as pd

# ============================================================
# COLUMN NAMES — unlabelled sheet has ONLY the abstract
# ============================================================
ABSTRACT_COL = "Abstract"
ID_COL       = None          # None = row index; MUST match the labelled run

# ============================================================
# RUN CONFIG
# NOTE: gemma-4-26b-a4b-it is served through the Gemini API. Record the run
# date alongside it — hosted aliases can change under you, a weaker
# reproducibility guarantee than pinned local Ollama tags.
# ============================================================
MODEL      = "gemma-4-26b-a4b-it"        # must match the labelled run exactly
OUTFILE    = "results_gemini_unlabelled.csv"
CATEGORIES = ["AI", "ML", "DL", "Unclear", "None"]


# ============================================================
# EVERYTHING FROM HERE TO THE PARSERS IS BYTE-IDENTICAL TO THE
# LABELLED NOTEBOOK. Any wording difference invalidates Cell E.
# ============================================================

CAT_BLOCK = """- AI: general artificial intelligence approaches, symbolic reasoning, expert systems, or work not specific to ML/DL techniques
- ML: classical machine learning — supervised/unsupervised learning, tree-based methods, SVMs, clustering, feature engineering
- DL: neural network architectures — CNNs, RNNs, transformers, deep reinforcement learning
- Unclear: the abstract does not provide enough information to assign a category individually. Like there is a mix of multiple categories in the abstract so unable to assign individual categories
- None: the paper is not about AI, ML or DL at all — these terms appear only incidentally or as background, and the paper's contribution lies in another field entirely"""


# ---- RULES: OLD (used by V2.1) ----
RULES_OLD = """Disambiguation rules:
- If the paper's core method is a neural network, classify as DL even though it is also ML and AI
- If multiple methods are compared, classify by the paper's primary contribution
- Use Unclear only when the abstract genuinely lacks the information needed, not when the paper is merely interdisciplinary
- Use None when AI/ML/DL are mentioned only as enabling technology or future possibility, and the paper itself contributes to a different field"""


# ---- RULES: NEW (used by V2.2 and V3) — adds the "primary approach" rule ----
RULES_NEW = RULES_OLD + """
- The context behind the classification is to identify the primary approach of the classification. Which is done by identifying the core concept the "methodology", "technical implementation" or the "review approach" of the paper follows to achieve its results, inference or conclusion"""


# ---- FEW-SHOT EXAMPLES (V3 only). Ordered None -> AI -> ML -> DL -> Unclear ----
FEWSHOT_BLOCK = """Worked examples:

--- EXAMPLE 1 ---
Abstract: Artificial intelligence offers great opportunities in critical care, particularly when a vast amount of continuously acquired physiological data is incorporated. High-quality, reliably labelled data are paramount for developing and training artificial intelligence methods. However, routinely recorded data in critical care are often noisy, and the sheer volume of high-resolution data is challenging to manage. Generalizable solutions for these problems are lacking, restricting progress. To address these barriers, we developed Vitabel, an open-source Python framework for post hoc loading, visualizing, aligning, and annotating medical time series. The framework provides sensible defaults and interactive components for efficient use in preconfigured workflows, while remaining flexible and extendable for custom analysis and annotation pipelines. It integrates seamlessly into Jupyter Notebooks, providing an interactive, customizable interface for visual interaction with the data. In this publication, we demonstrate its utility across three use cases. The code and exemplary data are provided as browser-based demos. Vitabel is freely available and published under the MIT license accompanying this publication.
REASONING: The paper's contribution is a Python framework for loading, visualising and annotating medical time series. Although AI is mentioned as motivation, no AI, ML or DL concept, method or architecture is developed or studied.
CATEGORY: None

--- EXAMPLE 2 ---
Abstract: Background: The integration of artificial intelligence (AI) into traditional Chinese medicine (TCM) research and development offers promising solutions to longstanding challenges in the field. These challenges include the complexity of TCM formulations, variability in quality control, and hurdles in global market acceptance. The unique synergy between AI technologies and TCM principles creates opportunities to enhance research efficiency, standardization, and innovation. Aim of review: This review aims to explore the applications and impact of AI across three critical stages of TCM development: drug design, pharmaceutical manufacturing, and market access. By summarizing the advancements and limitations in these areas, the review identifies the transformative potential of AI and proposes future directions for integrating AI with emerging technologies to advance TCM research and development (R&D). Key scientific concepts of review: AI has transformative potential in TCM development, addressing key challenges across various stages. In drug design, AI accelerates the identification of active compounds, optimizes formula composition, and models pharmacodynamic relationships to enhance innovation efficiency and precision. During pharmaceutical manufacturing, AI contributes to process optimization, quality control, and the standardization of TCM products, ensuring stable and scalable production. For market access, although no TCM developed by AI has entered the clinic, AI has played a role in comprehensive safety and efficacy assessments and simplified regulatory compliance in other drugs. By leveraging these advances and reviewing limitations, AI promotes the need to develop more integrated, more efficient, and more utilized methods in TCM R&D.
REASONING: The review discusses AI across TCM drug design, manufacturing, and market access without naming a single algorithm, architecture, or learning methodology anywhere. The treatment of AI remains at the level of general capability rather than any specific ML or DL technique.
CATEGORY: AI

--- EXAMPLE 3 ---
Abstract: Background: Understanding how the respiratory microbiota matures with age is key to improving poultry health and pathogen surveillance, yet the ecological processes shaping this transition remain elusive. We aimed to develop an interpretable machine-learning framework capable of identifying age-associated microbial signatures within the chicken nasal microbiota across heterogeneous datasets. Results: We compiled data from five independent chicken studies and normalized microbial abundances using Counts Per Million (CPM). To address dataset imbalance and ensure cross-study generalizability, we implemented SMOTE over-sampling and a Leave-One-Study-Out (LOSO) cross-validation framework. Within this architecture, we utilized Recursive Feature Elimination (RFE) to identify a stable consensus signature composed of taxa persisting in at least 70% of the iterations. We benchmarked five algorithms: Classification and Regression Trees (CART), k-nearest neighbors (kNN), Support Vector Machines (SVM), Random Forest (RF), and Extreme Gradient Boosting (XGBoost). RF emerged as the best model, achieving a balanced accuracy of 0.965 and a Kappa of 0.920. Consequently, the contribution of each feature was quantified through SHapley Additive exPlanations (SHAP) values on the selected RF model, enabling transparent interpretation of age-dependent microbial patterns. This approach distilled a compact set of predictive taxa, including Corynebacterium, Kocuria, and members of the Micrococcaceae. External validation with longitudinal samples from a Highly Pathogenic Avian Influenza Virus (HPAIV) infection confirmed full generalization, with all 57 samples from 22 chickens correctly classified even under viral-induced conditions. Conclusions: The proposed workflow combining LOSO-based feature selection, class-balancing, and interpretable machine learning provides a transferable framework for microbiota-based age inference.
REASONING: Every model in the paper is a classical algorithm of ML with no neural network anywhere, so it cannot sit at the deep learning tier. The rest of the work is feature engineering and validation design — normalisation, resampling, feature elimination, cross-validation — which is textbook machine learning practice. The poultry biology is just the subject matter the methods are applied to, and SHAP is only used afterwards to explain the chosen model.
CATEGORY: ML

--- EXAMPLE 4 ---
Abstract: Early detection and prediction of Depressive Symptoms is essential for improving mental health outcomes. This study proposes a hybrid deep learning and machine learning framework that utilizes tabular data collected from wearable devices, including sleep patterns, physical activity, and health-related indicators. Three ensemble learning models were used to identify influential predictors through the application of explainable artificial intelligence techniques such as SHAP and LIME. Based on the selected important features, two hybrid models were developed combining 1D-Convolutional Neural Network and Multi-layer Perceptron with LightGBM. The experimental results showed that models trained on selected features consistently outperformed those using the full feature set. The highest classification accuracy, 93.43%, was achieved by the multilayer perceptron model with LightGBM when trained on features selected by XGBoost. SHAP analysis highlighted the importance of features such as night sleep duration, age, and income responsibility, while LIME provided sample specific explanations that enhanced local interpretability. This framework enhances both the predictive performance and interpretability of depression prediction models and demonstrates the potential of wearable-derived behavior and physiological features as practical biomarkers for personalized risk assessment.
REASONING: The main subject of the paper involves neural network architecture and concepts like 1D-CNN and MLP. Although XGBoost and SHAP/LIME appear in the abstract, they are not the main subject of contribution — they are used only as tools supporting the aforementioned neural network architectures.
CATEGORY: DL

--- EXAMPLE 5 ---
Abstract: Artificial intelligence (AI) is reshaping education by enabling personalized learning and increasing student engagement. The rapid adoption of tools such as ChatGPT, however, raises questions about efficacy, academic integrity, and ethics. This study aims to fill a critical gap by providing a systematic comparative evaluation of classical, deep learning, and transformer-based models for sentiment analysis of ChatGPT-related educational discourse, identifying the best-performing approach, and deploying it in an explainable, user-friendly web application for non-technical stakeholders. This study gathered 236,275 tweets related to ChatGPT in education and implemented a systematic benchmarking process. A Naive Bayes model with TF-IDF vectorization served as the classical baseline. For the deep learning tier, Long Short-Term Memory (LSTM) and Bidirectional LSTM (Bi-LSTM) models with Word2Vec embeddings were used. The transformer tier was represented by a fine-tuned DistilBERT model. The fine-tuned DistilBERT achieved the highest accuracy at 98.81%. In contrast, the LSTM model achieved 95.40%, the Bi-LSTM reached 94.66%, and the Naive Bayes model recorded an accuracy of 81.77%. A Streamlit-based web application was developed that integrates LIME-based explainability.
REASONING: The study is layered across an AI system (ChatGPT) as its subject, classical ML (Naive Bayes, TF-IDF), and DL (LSTM, Bi-LSTM, DistilBERT), with each tier given comparable weight. No single category is identifiable as the primary contribution.
CATEGORY: Unclear"""

FEWSHOT_EXTRA_RULE = """- Use these examples to understand the context behind the classification. The primary approach or deciding factor of the classification is what core concept the "methodology", "technical implementation" or the "review approach" of the paper follows to achieve its results, inference or conclusion"""


# ============================================================
# COT INSTRUCTION BLOCK (shared by V2.1, V2.2, V3)
# ============================================================
COT_STEPS = """Work through this in three steps:
1. Identify the technical methods described in the text
2. Determine which method represents the paper's primary contribution
3. Assign the most specific applicable category

Respond in exactly this format:
REASONING: <two or three sentences>
CATEGORY: <one category name>"""

HEADER = """You are an expert in artificial intelligence and machine learning research.

IMPORTANT CONTEXT: These categories are hierarchically nested. Deep learning is a subset of machine learning, and machine learning is a subset of artificial intelligence. Assign the MOST SPECIFIC category that applies."""


# ============================================================
# PROMPT VERSIONS
#   V1   = definition-augmented zero-shot (no rules, no CoT)
#   V2.1 = CoT + OLD rules
#   V2.2 = CoT + NEW rules
#   V3   = CoT + few-shot + NEW rules
# ============================================================
PROMPTS = {

"V1": f"""You are classifying academic paper abstracts by their primary technical contribution.

Categories:
{CAT_BLOCK}

Assign the single category that best matches the paper's PRIMARY contribution.

Respond with only the category name.

{{text}}""",

"V2.1": f"""{HEADER}

Categories:
{CAT_BLOCK}

{RULES_OLD}

{COT_STEPS}

{{text}}""",

"V2.2": f"""{HEADER}

Categories:
{CAT_BLOCK}

{RULES_NEW}

{COT_STEPS}

{{text}}""",

"V3": f"""{HEADER}

Categories:
{CAT_BLOCK}

{RULES_NEW}
{FEWSHOT_EXTRA_RULE}

{FEWSHOT_BLOCK}

Now classify the following paper.

{COT_STEPS}

{{text}}""",
}

COT_VERSIONS = {"V2.1", "V2.2", "V3"}


# ============================================================
# INPUT BUILDER — abstract only
# ============================================================
MISSING_TOKENS = {"", "nan", "none", "null", "n/a", "na"}

def _blank(v):
    """True if the cell is empty, NaN-like, or an explicit 'no abstract' marker."""
    s = str(v).strip().lower()
    return s in MISSING_TOKENS or "no abstract available" in s


def build_input(row):
    """Return (text_block, source_used). source_used: abstract | missing"""
    if not _blank(row.get(ABSTRACT_COL)):
        return f"Abstract: {str(row[ABSTRACT_COL]).strip()}", "abstract"
    return None, "missing"


# ============================================================
# LABEL NORMALISATION
# Unused in this run (no manual labels), but Cell E needs it.
# ============================================================
LABEL_MAP = {
    "artificial intelligence": "AI", "ai": "AI",
    "machine learning": "ML",        "ml": "ML",
    "deep learning": "DL",           "dl": "DL",
    "unclear": "Unclear",
    "none": "None",
}

def normalise(label):
    return LABEL_MAP.get(str(label).strip().lower(), str(label).strip())


# ============================================================
# OUTPUT PARSING — three tiers, identical to the labelled run
# ============================================================
def _match(token):
    token = token.strip(".,:;*\"'`")
    for c in CATEGORIES:
        if token.lower() == c.lower():
            return c
    return "UNPARSEABLE"


# Match a category only as a standalone word, not inside another word
_CAT_RE = re.compile(r"\b(AI|ML|DL|Unclear|None)\b", re.IGNORECASE)

def extract_label(raw, version):
    """1) Prefer an explicit CATEGORY: line (any position).
       2) Otherwise try the first token.
       3) Otherwise take the LAST standalone category mention — models that
          reason before answering put the verdict at the end."""
    text = raw.strip()

    hits = re.findall(r"CATEGORY\s*:\s*\**\s*(\w+)", text, re.IGNORECASE)
    if hits:
        lab = _match(hits[-1])
        if lab != "UNPARSEABLE":
            return lab

    stripped = text.strip("*# \n")
    parts = stripped.split()
    if parts:
        lab = _match(parts[0])
        if lab != "UNPARSEABLE":
            return lab

    found = _CAT_RE.findall(text)
    if found:
        canon = {"ai": "AI", "ml": "ML", "dl": "DL",
                 "unclear": "Unclear", "none": "None"}
        return canon[found[-1].lower()]

    return "UNPARSEABLE"


# ============================================================
# FEW-SHOT LEAKAGE TAGGING
# ============================================================
FEWSHOT_FINGERPRINTS = [
    "vitabel, an open-source python framework",
    "integration of artificial intelligence (ai) into traditional chinese medicine",
    "age-associated microbial signatures within the chicken nasal microbiota",
    "1d-convolutional neural network and multi-layer perceptron with lightgbm",
    "236,275 tweets related to chatgpt in education",
]

def is_fewshot_row(row):
    a = str(row.get(ABSTRACT_COL, "")).lower()
    return any(fp in a for fp in FEWSHOT_FINGERPRINTS)


# ============================================================
# PRE-RUN AUDIT
# ============================================================
assert ABSTRACT_COL in df.columns, f"'{ABSTRACT_COL}' not in {list(df.columns)}"

sources = df.apply(lambda r: build_input(r)[1], axis=1)
src_counts = sources.value_counts()

print("=" * 55)
print(f"Rows: {len(df)}   Versions: {len(PROMPTS)}   Calls: {len(df) * len(PROMPTS)}")
print(f"Model: {MODEL}   (unlabelled run — leakage verification)")
print("=" * 55)
print(f"\n  abstract usable ........... {src_counts.get('abstract', 0)}")
print(f"  no-abstract record count = {src_counts.get('missing', 0)}")
print(f"\nFew-shot example rows matched: {df.apply(is_fewshot_row, axis=1).sum()} / 5")

# Prompt hashes MUST match the labelled notebook, or Cell E means nothing
print("\nPrompt hashes (must match the labelled notebook):")
for v, t in PROMPTS.items():
    print(f"  {v:<6} {hashlib.md5(t.encode()).hexdigest()[:12]}  ({len(t)} chars)")
print("=" * 55)


# ============================================================
# GEMINI CALLER
# Hosted APIs return transient 5xx/429 under load. Without retry these
# become permanent ERROR rows and are silently lost from the run.
# ============================================================
TRANSIENT = ("503", "502", "500", "429", "overloaded", "unavailable",
             "quota", "rate", "timeout", "deadline", "internal")

def call_model(prompt, model_name, retries=6):
    model = genai.GenerativeModel(model_name)
    for attempt in range(retries):
        try:
            r = model.generate_content(
                prompt,
                generation_config={"temperature": 0,        # deterministic
                                   "max_output_tokens": 400}
            )
            txt = (r.text or "").strip()
            if not txt:
                return "", "empty response (possible safety block)"
            return txt, ""
        except Exception as e:
            msg = str(e)
            if any(t in msg.lower() for t in TRANSIENT):
                wait = min(4 * (2 ** attempt), 120)
                print(f"  transient error — retry {attempt+1}/{retries} in {wait}s", flush=True)
                time.sleep(wait)
                continue
            return "", msg          # non-transient: fail immediately
    return "", f"failed after {retries} retries"

Rows: 300   Versions: 4   Calls: 1200
Model: gemma-4-26b-a4b-it   (unlabelled run — leakage verification)

  abstract usable ........... 297
  no-abstract record count = 3

Few-shot example rows matched: 2 / 5

Prompt hashes (must match the labelled notebook):
  V1     985a959e5b6b  (933 chars)
  V2.1   6d14df3d5ec4  (1803 chars)
  V2.2   c056e5303913  (2088 chars)
  V3     30057e1896ca  (11782 chars)


In [5]:
import sys

# ---- Resume support ----
if os.path.exists(OUTFILE):
    done_df = pd.read_csv(OUTFILE)
    done = set(zip(done_df["paper_id"].astype(str),
                   done_df["model_version"],
                   done_df["prompt_version"]))
    print(f"Resuming — {len(done)} rows already complete\n", flush=True)
else:
    done_df, done = pd.DataFrame(), set()

# ---- Fail fast if the API is unreachable ----
try:
    genai.GenerativeModel(MODEL).generate_content("test")
    print("✓ Gemini API reachable\n", flush=True)
except Exception as e:
    raise SystemExit(f"Gemini API unreachable — check the API key. ({e})")

rows, counter, skipped = [], 0, 0
total = len(df) * len(PROMPTS)
run_start = time.time()

# No manual label in this run, so no correctness mark
print(f"{'#':>6} {'VER':<6} {'SRC':<9} {'PRED':<12} {'SEC':>6} {'ETA':>7}")
print("-" * 55, flush=True)

for version, template in PROMPTS.items():
    for idx, r in df.iterrows():
        counter += 1
        pid = str(r[ID_COL]) if ID_COL else str(idx)

        if (pid, MODEL, version) in done:
            skipped += 1
            continue

        text_block, source = build_input(r)

        # Blank abstract — logged as NO_INPUT, never silently dropped
        if text_block is None:
            rows.append({
                "paper_id": pid, "model_version": MODEL, "prompt_version": version,
                "input_source": "missing", "predicted_label": "NO_INPUT",
                "raw_output": "", "elapsed_sec": 0.0,
                "error": "no abstract available",
                "is_fewshot_example": is_fewshot_row(r),
                "run_timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),
            })
            print(f"{counter:>6} {version:<6} {'missing':<9} {'NO_INPUT':<12}", flush=True)
            continue

        t0 = time.time()
        raw, err = call_model(template.format(text=text_block), MODEL)
        elapsed = round(time.time() - t0, 2)

        time.sleep(1)   # stay inside the free-tier requests-per-minute cap

        label = extract_label(raw, version) if raw else "ERROR"

        rows.append({
            "paper_id":           pid,
            "model_version":      MODEL,
            "prompt_version":     version,
            "input_source":       source,
            "predicted_label":    label,
            "raw_output":         raw,         # retained for error analysis
            "elapsed_sec":        elapsed,     # NB: includes network latency
            "error":              err,
            "is_fewshot_example": is_fewshot_row(r),
            "run_timestamp":      pd.Timestamp.now().isoformat(timespec="seconds"),
        })

        # ---- live progress ----
        processed = counter - skipped
        avg = (time.time() - run_start) / max(processed, 1)
        eta = (total - counter) * avg / 60
        print(f"{counter:>6} {version:<6} {source:<9} {label:<12} "
              f"{elapsed:>6.2f} {eta:>6.0f}m", flush=True)
        if err:
            print(f"       └─ ERROR: {err[:80]}", flush=True)

        # ---- flush every 10 rows so a disconnect costs little ----
        if len(rows) >= 10:
            pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).to_csv(OUTFILE, index=False)
            done_df = pd.read_csv(OUTFILE)
            rows = []

if rows:
    pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).to_csv(OUTFILE, index=False)

results = pd.read_csv(OUTFILE)
print("\n" + "=" * 55)
print(f"Complete — {len(results)} rows | {(time.time()-run_start)/60:.1f} min")
print(f"Errors: {results['error'].astype(str).ne('').sum()} | "
      f"Unparseable: {(results.predicted_label == 'UNPARSEABLE').sum()}", flush=True)

✓ Gemini API reachable

     # VER    SRC       PRED            SEC     ETA
-------------------------------------------------------
     1 V1     abstract  DL             8.43    189m
     2 V1     abstract  ML             9.32    197m


KeyboardInterrupt: 

In [5]:
# ============================================================
# Recover rows that exhausted their retries during the main run
# ============================================================
results = pd.read_csv(OUTFILE)
failed = results[results.predicted_label == "ERROR"]
print(f"Retrying {len(failed)} failed rows...", flush=True)

fixed = 0
for i, row in failed.iterrows():
    src = df.loc[int(row["paper_id"])]
    text_block, _ = build_input(src)
    if text_block is None:
        continue

    t0 = time.time()
    raw, err = call_model(PROMPTS[row["prompt_version"]].format(text=text_block), MODEL)
    if raw:
        results.loc[i, ["predicted_label", "raw_output", "elapsed_sec", "error"]] = [
            extract_label(raw, row["prompt_version"]), raw, round(time.time()-t0, 2), ""
        ]
        fixed += 1
    time.sleep(2)

results.to_csv(OUTFILE, index=False)
print(f"Recovered {fixed}/{len(failed)} rows", flush=True)

Retrying 11 failed rows...
Recovered 11/11 rows


In [6]:
results = pd.read_csv(OUTFILE)

print(f"Model: {MODEL}   (unlabelled run — no ground truth in this file)\n")

print("Predicted label distribution by version")
print(pd.crosstab(results.prompt_version, results.predicted_label), "\n")

print("Timing by version (supports the scalability argument)")
print(results.groupby("prompt_version").agg(
    n=("elapsed_sec", "size"),
    mean_sec=("elapsed_sec", "mean"),
    total_min=("elapsed_sec", lambda s: s.sum() / 60),
).round(2), "\n")

print("Failure counts")
print(results.predicted_label.value_counts().reindex(
    ["ERROR", "UNPARSEABLE", "NO_INPUT"]).fillna(0).astype(int))

Model: gemma-4-26b-a4b-it   (unlabelled run — no ground truth in this file)

Predicted label distribution by version
predicted_label  AI   DL   ML  NO_INPUT  Unclear
prompt_version                                  
V1               44   98  107         3       35
V2.1             45  123  108         3        7
V2.2             56  101  123         3        6
V3               81   87   95         3       20 

Timing by version (supports the scalability argument)
                  n  mean_sec  total_min
prompt_version                          
V1              300      9.07      45.34
V2.1            300      9.23      46.15
V2.2            300      9.22      46.10
V3              300     10.01      50.02 

Failure counts
predicted_label
ERROR           0
UNPARSEABLE     0
NO_INPUT       12
Name: count, dtype: int64


In [1]:
lab = pd.read_csv("results_gemini_unlabelled.csv")
unl = pd.read_csv(OUTFILE)

merged = lab.merge(
    unl[["paper_id", "model_version", "prompt_version", "predicted_label"]],
    on=["paper_id", "model_version", "prompt_version"],
    suffixes=("_labelled", "_unlabelled")
)
merged["identical"] = merged.predicted_label_labelled == merged.predicted_label_unlabelled

print(f"Matched rows: {len(merged)}\n")
print("Agreement between labelled and unlabelled runs:")
print(merged.groupby("prompt_version").identical.agg(n="size", agreement="mean").round(4))

overall = merged.identical.mean()
print(f"\nOverall agreement: {overall:.4f}")

if overall == 1.0:
    print("\n✓ Identical predictions across both runs — the manual classification "
          "column had no influence on model output.")
else:
    print(f"\n⚠️  {(~merged.identical).sum()} rows differ.")
    print(merged[~merged.identical][
        ["paper_id", "prompt_version",
         "predicted_label_labelled", "predicted_label_unlabelled"]].head(20))

from google.colab import files
files.download(OUTFILE)

NameError: name 'pd' is not defined